# 6. Alignment: kappa, chance and poles

The stand-in model calls a text toxic when it ends with "!". On the toy data that marks the right-wing side of politics and the women's side of gender for half the texts, so we expect a clear pattern.

In [1]:
import tempfile
from pathlib import Path

import pandas as pd

from perspective_audit import run_dataset
from perspective_audit.toy import toy_ratings, toy_spec

pd.set_option("display.width", 160)
result = run_dataset(toy_spec(), raw=toy_ratings())        # the toy dataset: 60 texts, 24 annotators, gender / politics / age

In [2]:
from perspective_audit import annotate, compute_kappa, compute_poles, load_model_inputs, save_au_tables
from perspective_audit.toy import standin_model

out = Path(tempfile.mkdtemp())
save_au_tables({"toy": result.au}, out)
inputs = load_model_inputs(out / "aposteriori_for_models.csv")
annotate(standin_model(), inputs, out, prompt=lambda texts, item: f"Rate each {item}:\n{texts}", run="baseline", item_words={"toy": "comment"}, show_progress=False)
given = pd.read_csv(out / "aposteriori_for_models.csv", dtype={"item_id": str})

In [3]:
kappa = compute_kappa({"toy": result}, out, given, reps=1_000, seed=42, models=["standin"], prompt="baseline")
kappa["by_group"][["characteristic", "group", "n_texts", "kappa", "ci_low", "ci_high", "se"]]

,characteristic,group,n_texts,kappa,ci_low,ci_high,se
0,gender,man,20,-1.0,-1.0,-0.6,0.1083
1,gender,woman,20,1.0,1.0,1.0,0.0000
2,politics,left,20,-1.0,-1.0,-0.6,0.1091
3,politics,right,20,1.0,1.0,1.0,0.0000


A kappa of 1 means the model sides with that group on every attributed text, -1 means it always takes the opposite side. The gap between two groups of a characteristic:

In [4]:
kappa["gaps"].iloc[:, :9]

,dataset,model,characteristic,prompt,group_a,group_b,n_texts_a,n_texts_b,gap
0,toy,standin,gender,baseline,man,woman,20,20,-2.0
1,toy,standin,politics,baseline,left,right,20,20,-2.0


## Poles

On the texts where the groups split, does the model side with the toxic pole or the civil pole?

In [5]:
poles = compute_poles({"toy": result}, out, given, reps=500, seed=42, models=["standin"])
poles["alignment"]

,dataset,characteristic,model,n_attributed,n_split,n_same,n_model_toxic,n_model_civil,share_model_toxic,sd,ci_low,ci_high,share_model_toxic_same,constant_model
0,toy,gender,standin,20,20,0,10,10,0.5,0.1087,0.30,0.700,NaN,False
1,toy,politics,standin,20,20,0,10,10,0.5,0.1097,0.30,0.750,NaN,False
2,toy,all (distinct texts),standin,40,40,0,20,20,0.5,0.0807,0.35,0.675,NaN,False
